In [17]:
import cv2
import numpy as np

# CONFIG - Change these variables as needed.
INPUT_VIDEO = "SampleB.mp4"
OUTPUT_VIDEO = "output_overlay.mp4"


In [18]:
#RED DETECTION - This function detects red objects in the frame using HSV color space and returns a binary mask.
def detect_red(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_red_1 = np.array([0, 120, 70])
    upper_red_1 = np.array([10, 255, 255])

    lower_red_2 = np.array([170, 120, 70])
    upper_red_2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red_1, upper_red_1)
    mask2 = cv2.inRange(hsv, lower_red_2, upper_red_2)

    mask = mask1 + mask2

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

In [19]:
# MAIN PROCESSING LOOP - This function processes the video frame by frame, detects the red ball, calculates its area and radius, and overlays this information on the video. It also tracks the time the ball is visible and the elapsed video time.

def run():
    cap = cv2.VideoCapture(INPUT_VIDEO)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

    frame_index = 0
    prev_area = None

    visible_start_frame = None  # NEW

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        mask = detect_red(frame)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # --- VIDEO TIME ---
        elapsed_time = frame_index / fps

        if contours:
            largest = max(contours, key=cv2.contourArea)

            if cv2.contourArea(largest) > 200:
                (x, y), radius = cv2.minEnclosingCircle(largest)

                center = (int(x), int(y))
                radius = int(radius)

                # --- START TRACKING TIME ---
                if visible_start_frame is None:
                    visible_start_frame = frame_index

                # --- VISIBLE TIME ---
                visible_time = (frame_index - visible_start_frame) / fps
                
                # --- AREA ---
                area = np.pi * radius * radius

                if prev_area is not None:
                    area = 0.8 * prev_area + 0.2 * area
                prev_area = area

                # --- DRAWINGS ---
                cv2.circle(frame, center, radius, (0, 255, 0), 2)
                cv2.circle(frame, center, 3, (0, 0, 0), -1)

                # --- TEXT ---
                cv2.putText(frame, f"Area: {int(area)} px",
                            (40, 840),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 0, 0), 2)

                cv2.putText(frame, f"Radius: {radius}px",
                            (40, 880),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 0, 0), 2)

                cv2.putText(frame, f"Video Time: {elapsed_time:.2f} s",
                            (40, 920),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 0, 0), 2)

                cv2.putText(frame, f"Visible Time: {visible_time:.2f} s",
                            (40, 960),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1, (0, 0, 0), 2)

        # Write frame
        out.write(frame)

        # Preview
        cv2.imshow("Tracking", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

        frame_index += 1

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print("Done. Output saved to:", OUTPUT_VIDEO)


if __name__ == "__main__":
    run()

Done. Output saved to: output_overlay.mp4
